# Day 014 — Exercise 5: Judge a RAG Answer

**Goal:** Implement `evaluate_rag_answer` — an LLM-as-judge function that scores RAG output for faithfulness and relevance.

**What you build:** A function that sends the question, answer, and context to a judge LLM and parses scores into a `RagEval` Pydantic model.

In [ ]:
import ollama
from pydantic import BaseModel, Field

## Your Task

Complete `evaluate_rag_answer` below. The `RagEval` model and `JUDGE_SYSTEM_PROMPT` are already defined — you only need to implement the function body.

Steps:
1. Build a `user_msg` string that includes the JSON schema, the joined context, the question, and the answer.
2. Call `ollama.chat` with `JUDGE_SYSTEM_PROMPT` as the system message and `user_msg` as the user message. Pass `format="json"`.
3. Parse `response["message"]["content"]` with `RagEval.model_validate_json()` and return the result.

## Your Implementation

In [ ]:
class RagEval(BaseModel):
    faithfulness: float = Field(description="0.0–1.0: every claim is supported by context")
    relevance:    float = Field(description="0.0–1.0: answer addresses the question")
    verdict:      str   = Field(description="PASS or FAIL")
    reason:       str   = Field(description="One sentence explaining the verdict")


JUDGE_SYSTEM_PROMPT = """You are an impartial evaluator of RAG answers.
Score the answer on two dimensions, each from 0.0 to 1.0:
- faithfulness: every claim in the answer must be directly supported by the provided context
- relevance: the answer must address the question asked
verdict: PASS if both scores >= 0.7, else FAIL.

Return ONLY a JSON object like this example:
{"faithfulness": 0.9, "relevance": 0.8, "verdict": "PASS", "reason": "All claims are supported by context."}"""


def evaluate_rag_answer(
    question: str,
    answer: str,
    context_chunks: list[str],
    model: str = "llama3.2",
) -> RagEval:
    """
    Score a RAG answer for faithfulness and relevance using an LLM judge.

    Args:
        question:       The original user question.
        answer:         The RAG pipeline's answer to evaluate.
        context_chunks: List of context strings provided to the RAG model.
        model:          Ollama model to use as the judge.

    Returns:
        RagEval with faithfulness, relevance, verdict, and reason fields.
    """
    sep = "\n\n---\n\n"
    context = sep.join(context_chunks)
    # TODO: build user_msg containing context, question, and answer
    # TODO: call ollama.chat with JUDGE_SYSTEM_PROMPT, user_msg, format="json"
    # TODO: parse response["message"]["content"] with RagEval.model_validate_json()
    pass

In [ ]:
def _run_checks():
    total = 5
    passed = 0

    context = [
        "Python was created by Guido van Rossum and first released in 1991.",
        "Python emphasises code readability and simplicity.",
    ]
    question = "Who created Python?"
    good_answer = "Python was created by Guido van Rossum [1]."

    # Run the judge once and reuse the result for all checks
    result = None
    try:
        result = evaluate_rag_answer(question, good_answer, context)
    except Exception as e:
        print(f"\u274c Could not call evaluate_rag_answer \u2014 {e}")
        print(f"\nScore: {passed}/{total}")
        return

    # Check 1: returns a RagEval instance
    try:
        assert isinstance(result, RagEval), f"Expected RagEval, got {type(result)}"
        passed += 1
        print("\u2705 Check 1: returns a RagEval instance")
    except Exception as e:
        print(f"\u274c Check 1: returns RagEval \u2014 {e}")

    # Check 2: faithfulness is in [0, 1]
    try:
        assert 0.0 <= result.faithfulness <= 1.0, \
            f"faithfulness {result.faithfulness} is outside [0, 1]"
        passed += 1
        print(f"\u2705 Check 2: faithfulness in [0,1] \u2014 got {result.faithfulness:.2f}")
    except Exception as e:
        print(f"\u274c Check 2: faithfulness range \u2014 {e}")

    # Check 3: relevance is in [0, 1]
    try:
        assert 0.0 <= result.relevance <= 1.0, \
            f"relevance {result.relevance} is outside [0, 1]"
        passed += 1
        print(f"\u2705 Check 3: relevance in [0,1] \u2014 got {result.relevance:.2f}")
    except Exception as e:
        print(f"\u274c Check 3: relevance range \u2014 {e}")

    # Check 4: verdict is PASS or FAIL
    try:
        assert result.verdict in ("PASS", "FAIL"), \
            f"verdict must be 'PASS' or 'FAIL', got {result.verdict!r}"
        passed += 1
        print(f"\u2705 Check 4: verdict is PASS or FAIL \u2014 got {result.verdict!r}")
    except Exception as e:
        print(f"\u274c Check 4: verdict value \u2014 {e}")

    # Check 5: faithful, relevant answer gets PASS
    try:
        assert result.verdict == "PASS", \
            f"A faithful, relevant answer should get PASS, got {result.verdict!r} " \
            f"(faith={result.faithfulness:.2f}, rel={result.relevance:.2f})"
        passed += 1
        print(f"\u2705 Check 5: correct answer gets PASS (faith={result.faithfulness:.2f}, rel={result.relevance:.2f})")
    except Exception as e:
        print(f"\u274c Check 5: PASS for good answer \u2014 {e}")

    if passed == total:
        print("\U0001f389 Exercise complete!")
    print(f"\nScore: {passed}/{total}")

_run_checks()

## Solution

<details>
<summary>Click to reveal</summary>

```python
class RagEval(BaseModel):
    faithfulness: float = Field(description="0.0–1.0: every claim is supported by context")
    relevance:    float = Field(description="0.0–1.0: answer addresses the question")
    verdict:      str   = Field(description="PASS or FAIL")
    reason:       str   = Field(description="One sentence explaining the verdict")


JUDGE_SYSTEM_PROMPT = """You are an impartial evaluator of RAG answers.
Score the answer on two dimensions, each from 0.0 to 1.0:
- faithfulness: every claim in the answer must be directly supported by the provided context
- relevance: the answer must address the question asked
verdict: PASS if both scores >= 0.7, else FAIL.

Return ONLY a JSON object like this example:
{"faithfulness": 0.9, "relevance": 0.8, "verdict": "PASS", "reason": "All claims are supported by context."}"""


def evaluate_rag_answer(
    question: str,
    answer: str,
    context_chunks: list[str],
    model: str = "llama3.2",
) -> RagEval:
    """Score a RAG answer for faithfulness and relevance using an LLM judge."""
    sep = "\n\n---\n\n"
    context = sep.join(context_chunks)
    user_msg = (
        f"Context:\n{context}\n\n"
        f"Question: {question}\n"
        f"Answer: {answer}\n\n"
        f"Evaluate and return JSON."
    )
    response = ollama.chat(
        model=model,
        messages=[
            {"role": "system", "content": JUDGE_SYSTEM_PROMPT},
            {"role": "user",   "content": user_msg},
        ],
        format="json",
    )
    return RagEval.model_validate_json(response["message"]["content"])
```

</details>